# 05 -- Model comparison

This notebook pulls together every model trained in notebooks 02-04 --
baseline CNN, ResNet-18, EfficientNet-B0, AST linear probe -- and
compares them on equal footing: same test set, same metrics, same split.
It assumes you've run the *full* configs (not just the fast in-notebook
demos), either via the CLI:

```
python -m watkins.train --config configs/baseline_cnn.yaml
python -m watkins.train --config configs/resnet18.yaml
python -m watkins.train --config configs/efficientnet_b0.yaml
python -m watkins.train --config configs/ast_linear_probe.yaml
python -m watkins.evaluate --checkpoint results/checkpoints/<run_name>_best.pt
python -m watkins.robustness --checkpoint results/checkpoints/<run_name>_best.pt
```

or by letting this project's own training pipeline populate `results/`.
If a checkpoint or metrics file is missing, the cells below will tell you
which command to run to produce it.

In [1]:
import os
import subprocess
import sys
import importlib.util

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # 1. Mount Drive and locate the project. Upload src/, configs/,
    #    pyproject.toml, requirements.txt to this path in My Drive first --
    #    NOT the multi-GB Watkins/ or results/ folders, those are handled
    #    separately below.
    from google.colab import drive
    drive.mount("/content/drive")

    PROJECT_DRIVE_PATH = "/content/drive/MyDrive/ShipsEAR"  # <-- edit if you used a different path
    if not os.path.exists(f"{PROJECT_DRIVE_PATH}/src/watkins"):
        raise FileNotFoundError(
            f"Expected the project's src/ folder at {PROJECT_DRIVE_PATH}/src on Google Drive.\n"
            "Upload src/, configs/, pyproject.toml, and requirements.txt there "
            "(skip the multi-GB Watkins/ and results/ folders), or edit "
            "PROJECT_DRIVE_PATH above to match where you put them."
        )
    sys.path.insert(0, f"{PROJECT_DRIVE_PATH}/src")

    # 2. Install whatever Colab's base image doesn't already have. Deliberately
    #    does NOT touch torch/torchaudio/torchvision -- Colab's preinstalled
    #    versions are already matched to its GPU + CUDA build, and reinstalling
    #    this project's CPU-only wheels here would silently disable the GPU.
    needed = ["transformers", "timm", "soundfile", "datasets", "huggingface_hub", "pyarrow"]
    missing = [pkg for pkg in needed if importlib.util.find_spec(pkg) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

    # 3. Data goes on fast local/ephemeral disk (re-derivable from the public
    #    Hugging Face source, no reason to pay Drive's slow random-I/O tax on
    #    thousands of per-epoch file reads); results go on Drive so trained
    #    checkpoints/metrics survive a runtime disconnect.
    os.environ["WATKINS_DATA_ROOT"] = "/content/watkins_data"
    os.environ["WATKINS_RESULTS_ROOT"] = f"{PROJECT_DRIVE_PATH}/results"

    from watkins.data import DATA_ROOT
    if not (DATA_ROOT / "manifest.csv").exists():
        print("Materializing the Watkins dataset locally -- one-time per Colab runtime, ~10-15 min...")
        subprocess.run([sys.executable, "-m", "watkins.prepare_data"], check=True)
else:
    sys.path.insert(0, "../src")

import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from watkins.data import CLASS_IDS, CLASS_INFO
from watkins.models import build_model
from watkins.utils import count_parameters, results_root

RESULTS = results_root()
RUN_NAMES = ["baseline_cnn", "resnet18", "efficientnet_b0", "ast_linear_probe"]
MODEL_KEY = {"baseline_cnn": "baseline_cnn", "resnet18": "resnet18",
             "efficientnet_b0": "efficientnet_b0", "ast_linear_probe": "ast"}


/home/kj/Documents/CodeProjects/ShipsEAR/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Headline comparison table: accuracy, macro-F1, and parameter budget

Macro-F1 (unweighted average of per-class F1) matters more than accuracy
here -- recall the severe long-tail imbalance from notebook 00 (killer
whale alone is ~28% of all clips). A model that leans on the two or three
biggest species can post decent accuracy while doing badly on the ~50
species you'd actually care about identifying reliably.

In [2]:
rows = []
for run_name in RUN_NAMES:
    metrics_path = RESULTS / "metrics" / f"{run_name}_eval.json"
    if not metrics_path.exists():
        print(f"missing: {metrics_path} -- run `python -m watkins.evaluate --checkpoint "
              f"results/checkpoints/{run_name}_best.pt` first")
        continue
    with open(metrics_path) as f:
        m = json.load(f)
    model, _ = build_model(MODEL_KEY[run_name], num_classes=54)
    rows.append(dict(
        model=run_name,
        accuracy=m["accuracy"],
        macro_f1=m["macro_f1"],
        weighted_f1=m["weighted_f1"],
        params=count_parameters(model),
    ))

comparison = pd.DataFrame(rows, columns=["model", "accuracy", "macro_f1", "weighted_f1", "params"]).set_index("model")
comparison


missing: ../results/metrics/baseline_cnn_eval.json -- run `python -m watkins.evaluate --checkpoint results/checkpoints/baseline_cnn_best.pt` first
missing: ../results/metrics/resnet18_eval.json -- run `python -m watkins.evaluate --checkpoint results/checkpoints/resnet18_best.pt` first
missing: ../results/metrics/efficientnet_b0_eval.json -- run `python -m watkins.evaluate --checkpoint results/checkpoints/efficientnet_b0_best.pt` first
missing: ../results/metrics/ast_linear_probe_eval.json -- run `python -m watkins.evaluate --checkpoint results/checkpoints/ast_linear_probe_best.pt` first


,accuracy,macro_f1,weighted_f1,params
model,,,,


In [3]:
if comparison.empty:
    print("No evaluated checkpoints yet -- run the commands printed above first.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    comparison[["accuracy", "macro_f1"]].plot(kind="bar", ax=axes[0])
    axes[0].set_ylim(0, 1)
    axes[0].set_title("accuracy vs. macro-F1 by model")
    axes[0].tick_params(axis="x", rotation=30)

    axes[1].scatter(comparison["params"], comparison["macro_f1"], s=80)
    for name, row in comparison.iterrows():
        axes[1].annotate(name, (row["params"], row["macro_f1"]), fontsize=8,
                          xytext=(5, 5), textcoords="offset points")
    axes[1].set_xscale("log")
    axes[1].set_xlabel("parameters (log scale)")
    axes[1].set_ylabel("macro F1")
    axes[1].set_title("capacity vs. performance")
    plt.tight_layout()
    plt.show()


No evaluated checkpoints yet -- run the commands printed above first.


Look at the second plot in particular: does more capacity actually buy
more accuracy on ~15,000 clips spread over 54 imbalanced species, or does
it plateau (or worse, dip once overfitting sets in)? That answer is
dataset-size-dependent, not a fact about any architecture in the
abstract -- it's the answer *for this dataset*.

## 2. Per-class F1: where does each architecture actually struggle?

`results/SUMMARY.md` (regenerated by `python -m watkins.summarize`) shows
this same table for the best-represented species, in markdown form.

In [4]:
first_metrics_path = next((RESULTS / "metrics" / f"{r}_eval.json" for r in RUN_NAMES
                            if (RESULTS / "metrics" / f"{r}_eval.json").exists()), None)
top_species = []
if first_metrics_path is not None:
    with open(first_metrics_path) as f:
        m0 = json.load(f)
    support = {name: rep["support"] for name, rep in m0["report"].items()
               if name not in ("accuracy", "macro avg", "weighted avg")}
    top_species = [n for n, _ in sorted(support.items(), key=lambda kv: -kv[1])[:15]]

per_class_rows = {}
for run_name in RUN_NAMES:
    metrics_path = RESULTS / "metrics" / f"{run_name}_eval.json"
    if not metrics_path.exists():
        continue
    with open(metrics_path) as f:
        m = json.load(f)
    per_class_rows[run_name] = [m["report"].get(sp, {}).get("f1-score", np.nan) for sp in top_species]

per_class_df = pd.DataFrame(per_class_rows, index=top_species)
per_class_df


""


In [5]:
if per_class_df.empty:
    print("No evaluated checkpoints yet -- run the commands from section 1 first.")
else:
    ax = per_class_df.plot(kind="bar", figsize=(11, 4.5))
    ax.set_ylabel("F1 score")
    ax.set_xlabel("species (top 15 by test-set support)")
    ax.set_ylim(0, 1)
    ax.set_title("per-class F1 across architectures")
    ax.tick_params(axis="x", rotation=60)
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()


No evaluated checkpoints yet -- run the commands from section 1 first.


A pattern worth specifically checking: is the same species hardest across
every architecture, not just one? If so, that's evidence the difficulty
is in the *data* (few tapes -- see `docs/class_reference.md` -- means
less acoustic variety to learn from) rather than any one model being
weak.

## 3. The leakage gap, at full scale

Notebook 02 demonstrated the leaky-split effect with a 5-epoch, 30%-data
demo. If you ran the full comparison
(`configs/baseline_cnn.yaml` on `--split-mode clip_random` vs. the
default `tape_grouped`), compare the two here:

In [6]:
honest_path = RESULTS / "metrics" / "baseline_cnn_eval.json"
leaky_path = RESULTS / "metrics" / "baseline_cnn_leak_check_full_eval.json"

if honest_path.exists() and leaky_path.exists():
    with open(honest_path) as f:
        honest = json.load(f)
    with open(leaky_path) as f:
        leaky = json.load(f)
    print(f"tape_grouped (honest) split: acc={honest['accuracy']:.3f}  macro_f1={honest['macro_f1']:.3f}")
    print(f"clip_random (leaky) split:   acc={leaky['accuracy']:.3f}  macro_f1={leaky['macro_f1']:.3f}")
    print(f"accuracy inflation from leakage: {leaky['accuracy'] - honest['accuracy']:+.3f}")
else:
    print("Run: python -m watkins.train --config configs/baseline_cnn.yaml --run-name "
          "baseline_cnn_leak_check_full --split-mode clip_random")
    print("then: python -m watkins.evaluate --checkpoint "
          "results/checkpoints/baseline_cnn_leak_check_full_best.pt")


Run: python -m watkins.train --config configs/baseline_cnn.yaml --run-name baseline_cnn_leak_check_full --split-mode clip_random
then: python -m watkins.evaluate --checkpoint results/checkpoints/baseline_cnn_leak_check_full_best.pt


Whatever number you get, that gap is not noise -- it's the same
architecture, same training budget, same everything except which clips
ended up in train vs. test. It's a concrete, measured answer to "how much
should I discount a bioacoustic classification accuracy number if I don't
know how the underlying data was split."

## Exercises

1. Which model has the best macro-F1-per-parameter ratio? Is it the one
   you'd actually deploy if compute/memory were constrained (e.g. an
   edge sensor on a buoy)?
2. Look at each model's confusion matrix figure
   (`results/figures/<run_name>_confusion.png`). Is the *same* species
   pair confused across all four architectures, or does each model have
   its own failure pattern? What would that tell you about whether the
   confusion is a data limitation vs. a model limitation?
3. Repeat the leakage-gap comparison for one more model (e.g. ResNet-18).
   Is the leakage inflation similar in size, or does a bigger/pretrained
   model "exploit" the leaky split more (memorizing tape-specific noise
   floors more easily) than the small baseline CNN?